# *Perkinsus marinus* infections
We're wondering if there is evidence of any low lying *P. marinus* infections in these oysters. To do this, I'm going to map my sequences to the [*P. marinus* genome](https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000006405.1/)

A lot of this code will be run through bash jobs

## 1. get *P. marinus* genome

In [ ]:
# get the genome
datasets download genome accession GCF_000006405.1 --include genome

the genome path is now: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/perkinsus_genome/ncbi_dataset/data/GCF_000006405.1/GCF_000006405.1_JCVI_PMG_1.0_genomic.fna`

## 2. align oyster RNA-sequences to *P. marinus* genome
submit as a job

In [ ]:
#!/bin/bash
#SBATCH --job-name=pmarinus_alignment
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH -p cpu
#SBATCH -t 1:00:00
#SBATCH -o pmarinus_alignment_%j.log
#SBATCH --mail-type=END,FAIL

#---------load modules---------#

module load bowtie2/2.5.2
module load samtools

#---------set paths---------#

cd /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmarinus_alignment

PMAR="/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/perkinsus_genome/ncbi_dataset/data/GCF_000006405.1/GCF_000006405.1_JCVI_PMG_1.0_genomic.fna"

INPUT="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all"

OUTPUT="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmarinus_alignment"

reference_index="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmar_reference_index"

mkdir -p "${OUTPUT}"

#---------commands---------#

# Build index
bowtie2-build "${PMAR}" "${reference_index}"

# Align samples
for R1 in "${INPUT}"/*_gi_1_val_1.fq.gz; do

    SAMPLE=$(basename "$R1" _gi_1_val_1.fq.gz)

    R2="${INPUT}/${SAMPLE}_gi_2_val_2.fq.gz"

    if [[ ! -f "$R2" ]]; then
        echo "Warning: R2 file not found: $R2"
        continue
    fi

    echo "======================================"
    echo "Aligning ${SAMPLE}"
    echo "R1: ${R1}"
    echo "R2: ${R2}"
    echo "======================================"

# run bowtie
    bowtie2 \
        --very-sensitive-local \
        --threads "${SLURM_CPUS_PER_TASK}" \ # use multiple threads
        -x "${reference_index}" \
        -1 "${R1}" \
        -2 "${R2}" \
        2> "${OUTPUT}/${SAMPLE}_bowtie2.log" \ # write output log
# immediately convert to SAM files
    | samtools view -bS - \
    | samtools sort -o "${OUTPUT}/${SAMPLE}_alignment.sorted.bam" 

    samtools index "${OUTPUT}/${SAMPLE}_alignment.sorted.bam"

done